# Combinatorial Difficulty Explorer

This notebook creates small, feasible scheduling instances that are intended to be difficult because of combinatorial structure rather than raw size.

The key idea is different from the large Alibaba stress notebook: here we reduce cluster capacity and job count, then create competition for attractive energy slots. Jobs are flexible enough to be scheduled outside the renewable/cheap block, but many of them can also use that block, so the optimizer must decide which jobs deserve the best slots.

The generator also creates multi-GPU jobs. This creates packing conflicts: for example, one free GPU may be unusable for a job that requires two GPUs.


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from itertools import product
import json
from pathlib import Path
import sys
import time
from typing import Any

import numpy as np
import pandas as pd
from gurobipy import GRB

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ModelConfig
from src.data.scenarios import DEFAULT_GPU_POWER_KW
from src.data.validation import validate_clusters, validate_hourly_inputs, validate_jobs
from src.evaluation.metrics import compute_summary_metrics
from src.evaluation.results import extract_cluster_hourly_results, extract_hourly_results, extract_schedule
from src.milp.gurobi_model import build_milp_model


## Difficulty Knobs

The most important knobs are:

- `target_gpu_utilization`: total GPU-slot demand divided by total GPU-slot capacity.
- `cheap_block_pressure`: GPU-slot demand from jobs whose windows overlap the cheap block divided by cheap-block GPU capacity.
- `cheap_window_share`: share of jobs whose windows are built to overlap the cheap block while still allowing execution outside it.
- `restricted_gpu_share`: share of jobs restricted to one GPU type.
- `multi_gpu_share`: share of jobs requiring more than one GPU.
- `window_min` and `window_max`: flexibility range. Too small becomes feasibility-hard; too large can become easy.


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "experiments" / "combinatorial_difficulty"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SLOT_MINUTES = 15
DELTA_T = SLOT_MINUTES / 60
HORIZON_SLOTS = 16
TIME_LIMIT_SECONDS = 60
MIP_GAP = 0.01

BASE_SEED = 101
REPLICATIONS = 5

# Debug override example:
# REPLICATIONS = 1

PUE = 1.20
RENEWABLE_PRICE = 35.0
PEAK_PRICE = 800.0
BASELINE_LOAD_MW = 0.0

SAVE_SELECTED_SOLUTIONS = True


In [ ]:
@dataclass(frozen=True)
class DifficultyConfig:
    """Parameters controlling one small combinatorial difficulty instance."""

    target_gpu_utilization: float = 0.70
    cheap_block_pressure: float = 1.40
    cheap_window_share: float = 0.75
    restricted_gpu_share: float = 0.35
    multi_gpu_share: float = 0.30
    window_min: int = 4
    window_max: int = 10
    cheap_block_start: int = 5
    cheap_block_length: int = 5
    max_jobs: int = 60
    price_contrast: float = 1.50
    contracted_power_ratio: float = 0.70
    cpu_per_gpu: float = 8.0
    memory_gb_per_gpu: float = 48.0
    cpu_jitter: float = 0.20
    memory_jitter: float = 0.20
    power_jitter: float = 0.10

    @property
    def cheap_block_end(self) -> int:
        """Return the exclusive end slot of the cheap block."""

        return self.cheap_block_start + self.cheap_block_length


SWEEP_CONFIGS = [
    DifficultyConfig(target_gpu_utilization=util, cheap_block_pressure=pressure, restricted_gpu_share=restricted, multi_gpu_share=multi)
    for util, pressure, restricted, multi in product(
        [0.60, 0.75, 0.85],
        [1.10, 1.50, 1.90],
        [0.25, 0.50],
        [0.20, 0.45],
    )
]

len(SWEEP_CONFIGS)


## Small Cluster Definition

This cluster is intentionally reduced. The purpose is to keep the number of binary variables small while preserving GPU-type heterogeneity and packing conflicts.


In [ ]:
def build_small_clusters() -> pd.DataFrame:
    """Build a reduced multi-GPU-type cluster table for combinatorial tests."""

    rows = []
    for gpu_type, gpu_count in {"G2": 4, "T4": 4, "V100M32": 2, "P100": 2}.items():
        power_capacity_kw = gpu_count * DEFAULT_GPU_POWER_KW[gpu_type]
        rows.append(
            {
                "cluster_id": f"cluster_{gpu_type.lower()}",
                "cluster_role": "small_gpu_type_pool",
                "capacity_kw": power_capacity_kw,
                "capacity": power_capacity_kw / 1000.0,
                "power_capacity_kw": power_capacity_kw,
                "gpu_type": gpu_type,
                "gpu_count": gpu_count,
                "gpu_capacity": gpu_count,
                "cpu_capacity": gpu_count * 8.0,
                "memory_capacity_gb": gpu_count * 48.0,
            }
        )
    clusters = pd.DataFrame(rows)
    validate_clusters(clusters)
    return clusters


clusters_df = build_small_clusters()
clusters_df


## Energy Landscape

The cheap block has lower grid prices and higher renewable availability, but not enough capacity for every job whose window touches it. Jobs can be scheduled outside that block, so the instance is not simply a feasibility test.


In [ ]:
def build_energy_inputs(config: DifficultyConfig, clusters: pd.DataFrame) -> tuple[pd.DataFrame, ModelConfig]:
    """Build slot-level energy inputs and scalar MILP config for one difficulty setting."""

    total_it_capacity = float(clusters["capacity"].sum())
    cheap_slots = set(range(config.cheap_block_start, config.cheap_block_end))
    rows = []
    for slot in range(HORIZON_SLOTS):
        is_cheap = slot in cheap_slots
        grid_price = 70.0 / config.price_contrast if is_cheap else 70.0 * config.price_contrast
        renewable_available = (0.55 if is_cheap else 0.05) * total_it_capacity * PUE
        rows.append(
            {
                "hour": slot,
                "renewable_available": renewable_available,
                "grid_price": grid_price,
                "baseline_load": BASELINE_LOAD_MW,
                "pue": PUE,
            }
        )
    hourly = pd.DataFrame(rows)
    validate_hourly_inputs(hourly)
    model_config = ModelConfig(
        contracted_power=config.contracted_power_ratio * total_it_capacity * PUE,
        renewable_price=RENEWABLE_PRICE,
        peak_price=PEAK_PRICE,
        delta_t=DELTA_T,
        pue=PUE,
    )
    return hourly, model_config


hourly_df, model_config = build_energy_inputs(DifficultyConfig(), clusters_df)
hourly_df


## Feasible But Difficult Instance Generator

The generator uses a hidden feasible placement only to guarantee that at least one schedule exists. It then gives many jobs windows that overlap the cheap block and extend outside it. That creates flexibility and cost tradeoffs instead of only forcing jobs into the renewable period.


In [ ]:
def interval_intersects(start: int, duration: int, slot: int) -> bool:
    """Return whether a job active interval covers a slot."""

    return start <= slot < start + duration


def window_intersects_block(earliest: int, latest: int, duration: int, config: DifficultyConfig) -> bool:
    """Return whether at least one feasible start overlaps the cheap block."""

    for start in range(earliest, latest + 1):
        if start < config.cheap_block_end and start + duration > config.cheap_block_start:
            return True
    return False


def current_gpu_hours(jobs: list[dict[str, Any]]) -> int:
    """Return total GPU-slot demand for generated jobs."""

    return int(sum(job["gpu_count_required"] * job["duration"] for job in jobs))


def cheap_window_gpu_hours(jobs: list[dict[str, Any]], config: DifficultyConfig) -> int:
    """Return GPU-slot demand from jobs with windows overlapping the cheap block."""

    return int(
        sum(
            job["gpu_count_required"] * job["duration"]
            for job in jobs
            if window_intersects_block(job["earliest_start"], job["latest_start"], job["duration"], config)
        )
    )


def choose_job_gpu_requirement(rng: np.random.Generator, config: DifficultyConfig, clusters: pd.DataFrame) -> tuple[int, str]:
    """Sample GPU count and GPU-type restriction for one candidate job."""

    if rng.random() < config.multi_gpu_share:
        gpu_count = int(rng.choice([2, 3], p=[0.75, 0.25]))
    else:
        gpu_count = 1

    feasible_types = clusters.loc[clusters["gpu_count"] >= gpu_count, "gpu_type"].tolist()
    if rng.random() < config.restricted_gpu_share:
        gpu_type_required = str(rng.choice(feasible_types))
    else:
        gpu_type_required = ""
    return gpu_count, gpu_type_required


def compatible_clusters_for_job(gpu_count: int, gpu_type_required: str, clusters: pd.DataFrame) -> list[str]:
    """Return cluster ids that can run a job with the sampled GPU requirement."""

    allowed_types = set(clusters["gpu_type"]) if gpu_type_required == "" else {gpu_type_required}
    compatible = clusters[(clusters["gpu_type"].isin(allowed_types)) & (clusters["gpu_count"] >= gpu_count)]
    return compatible["cluster_id"].tolist()


def can_place_hidden(
    cluster: str,
    start: int,
    duration: int,
    gpu_count: int,
    cpu: float,
    memory: float,
    power: float,
    usage: dict[str, dict[str, np.ndarray]],
    clusters_by_id: dict[str, dict[str, Any]],
) -> bool:
    """Check hidden placement feasibility for all enforced resources."""

    end = start + duration
    cluster_data = clusters_by_id[cluster]
    return bool(
        np.all(usage[cluster]["gpu"][start:end] + gpu_count <= cluster_data["gpu_count"])
        and np.all(usage[cluster]["cpu"][start:end] + cpu <= cluster_data["cpu_capacity"])
        and np.all(usage[cluster]["memory"][start:end] + memory <= cluster_data["memory_capacity_gb"])
        and np.all(usage[cluster]["power"][start:end] + power <= cluster_data["capacity"])
    )


def reserve_hidden(
    cluster: str,
    start: int,
    duration: int,
    gpu_count: int,
    cpu: float,
    memory: float,
    power: float,
    usage: dict[str, dict[str, np.ndarray]],
) -> None:
    """Reserve hidden resource usage after accepting a generated job."""

    end = start + duration
    usage[cluster]["gpu"][start:end] += gpu_count
    usage[cluster]["cpu"][start:end] += cpu
    usage[cluster]["memory"][start:end] += memory
    usage[cluster]["power"][start:end] += power


def make_window(hidden_start: int, duration: int, should_overlap_cheap: bool, rng: np.random.Generator, config: DifficultyConfig) -> tuple[int, int]:
    """Build a flexible window that includes the hidden feasible start."""

    latest_possible = HORIZON_SLOTS - duration
    if should_overlap_cheap:
        anchor_left = min(hidden_start, config.cheap_block_start)
        anchor_right = max(hidden_start, config.cheap_block_end - duration)
        earliest = max(0, anchor_left - int(rng.integers(0, 3)))
        latest = min(latest_possible, anchor_right + int(rng.integers(0, 3)))
    else:
        target_width = int(rng.integers(config.window_min, config.window_max + 1))
        left_slack = int(rng.integers(0, target_width))
        earliest = max(0, hidden_start - left_slack)
        latest = min(latest_possible, earliest + target_width - 1)
        if hidden_start > latest:
            shift = hidden_start - latest
            earliest = max(0, earliest + shift)
            latest = min(latest_possible, latest + shift)

    if latest < hidden_start:
        latest = hidden_start
    if earliest > hidden_start:
        earliest = hidden_start

    width = latest - earliest + 1
    if width < config.window_min:
        missing = config.window_min - width
        earliest = max(0, earliest - missing // 2 - missing % 2)
        latest = min(latest_possible, latest + missing // 2)
    if latest < hidden_start:
        latest = hidden_start
    if earliest > hidden_start:
        earliest = hidden_start
    return int(earliest), int(latest)


def generate_difficulty_instance(config: DifficultyConfig, seed: int, clusters: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Generate one small feasible instance with controlled optimization difficulty knobs."""

    rng = np.random.default_rng(seed)
    clusters_by_id = clusters.set_index("cluster_id").to_dict(orient="index")
    usage = {
        cluster_id: {
            "gpu": np.zeros(HORIZON_SLOTS),
            "cpu": np.zeros(HORIZON_SLOTS),
            "memory": np.zeros(HORIZON_SLOTS),
            "power": np.zeros(HORIZON_SLOTS),
        }
        for cluster_id in clusters_by_id
    }

    total_gpu_capacity_slots = int(clusters["gpu_count"].sum() * HORIZON_SLOTS)
    cheap_gpu_capacity_slots = int(clusters["gpu_count"].sum() * config.cheap_block_length)
    target_gpu_slots = int(np.ceil(config.target_gpu_utilization * total_gpu_capacity_slots))
    target_cheap_gpu_slots = int(np.ceil(config.cheap_block_pressure * cheap_gpu_capacity_slots))

    jobs: list[dict[str, Any]] = []
    hidden_rows: list[dict[str, Any]] = []
    attempts = 0
    max_attempts = 20_000

    while (
        (current_gpu_hours(jobs) < target_gpu_slots or cheap_window_gpu_hours(jobs, config) < target_cheap_gpu_slots)
        and len(jobs) < config.max_jobs
        and attempts < max_attempts
    ):
        attempts += 1
        duration = int(rng.choice([1, 2, 3, 4], p=[0.20, 0.35, 0.30, 0.15]))
        gpu_count, gpu_type_required = choose_job_gpu_requirement(rng, config, clusters)
        compatible = compatible_clusters_for_job(gpu_count, gpu_type_required, clusters)
        if not compatible:
            continue

        per_gpu_cpu = config.cpu_per_gpu * rng.uniform(1.0 - config.cpu_jitter, 1.0 + config.cpu_jitter)
        per_gpu_memory = config.memory_gb_per_gpu * rng.uniform(1.0 - config.memory_jitter, 1.0 + config.memory_jitter)
        representative_gpu_type = gpu_type_required or str(rng.choice(clusters["gpu_type"].tolist()))
        per_gpu_power_kw = DEFAULT_GPU_POWER_KW[representative_gpu_type] * rng.uniform(1.0 - config.power_jitter, 1.0 + config.power_jitter)
        cpu = float(gpu_count * per_gpu_cpu)
        memory = float(gpu_count * per_gpu_memory)
        power = float(gpu_count * per_gpu_power_kw / 1000.0)

        should_overlap_cheap = rng.random() < config.cheap_window_share or cheap_window_gpu_hours(jobs, config) < target_cheap_gpu_slots
        if should_overlap_cheap:
            start_low = max(0, config.cheap_block_start - config.window_max)
            start_high = min(HORIZON_SLOTS - duration, config.cheap_block_end + config.window_max)
        else:
            start_low = 0
            start_high = HORIZON_SLOTS - duration
        if start_high < start_low:
            continue

        candidate_clusters = list(compatible)
        rng.shuffle(candidate_clusters)
        candidate_starts = list(range(start_low, start_high + 1))
        rng.shuffle(candidate_starts)

        placed = False
        for cluster in candidate_clusters:
            for hidden_start in candidate_starts:
                if can_place_hidden(cluster, hidden_start, duration, gpu_count, cpu, memory, power, usage, clusters_by_id):
                    earliest, latest = make_window(hidden_start, duration, should_overlap_cheap, rng, config)
                    job_id = f"job_{len(jobs):03d}"
                    jobs.append(
                        {
                            "job_id": job_id,
                            "category": "gpu_restricted" if gpu_type_required else "gpu_unrestricted",
                            "workload_family": "combinatorial_difficulty",
                            "duration": duration,
                            "power": power,
                            "earliest_start": earliest,
                            "latest_start": latest,
                            "gpu_type_required": gpu_type_required,
                            "gpu_count_required": gpu_count,
                            "cpu_required": cpu,
                            "memory_required_gb": memory,
                        }
                    )
                    hidden_rows.append(
                        {
                            "job_id": job_id,
                            "hidden_cluster": cluster,
                            "hidden_start": hidden_start,
                            "duration": duration,
                        }
                    )
                    reserve_hidden(cluster, hidden_start, duration, gpu_count, cpu, memory, power, usage)
                    placed = True
                    break
            if placed:
                break

    jobs_df = pd.DataFrame(jobs)
    hidden_df = pd.DataFrame(hidden_rows)
    validate_jobs(jobs_df)
    return jobs_df, hidden_df


## Instance Diagnostics


In [ ]:
def instance_diagnostics(jobs_df: pd.DataFrame, config: DifficultyConfig, clusters: pd.DataFrame) -> dict[str, float]:
    """Compute structural metrics before solving an instance."""

    total_gpu_capacity_slots = float(clusters["gpu_count"].sum() * HORIZON_SLOTS)
    cheap_gpu_capacity_slots = float(clusters["gpu_count"].sum() * config.cheap_block_length)
    job_gpu_slots = float((jobs_df["gpu_count_required"] * jobs_df["duration"]).sum())
    cheap_overlap_gpu_slots = float(cheap_window_gpu_hours(jobs_df.to_dict(orient="records"), config))
    window_width = jobs_df["latest_start"] - jobs_df["earliest_start"] + 1
    global_utilization = job_gpu_slots / total_gpu_capacity_slots
    cheap_pressure = cheap_overlap_gpu_slots / cheap_gpu_capacity_slots
    return {
        "num_jobs": int(len(jobs_df)),
        "job_gpu_slots": job_gpu_slots,
        "global_gpu_slot_utilization": global_utilization,
        "cheap_block_pressure_actual": cheap_pressure,
        "global_utilization_target_met": bool(global_utilization >= config.target_gpu_utilization),
        "cheap_pressure_target_met": bool(cheap_pressure >= config.cheap_block_pressure),
        "restricted_gpu_share_actual": float((jobs_df["gpu_type_required"].astype(str).str.len() > 0).mean()),
        "multi_gpu_share_actual": float((jobs_df["gpu_count_required"] > 1).mean()),
        "mean_duration_slots": float(jobs_df["duration"].mean()),
        "median_window_width_slots": float(window_width.median()),
        "mean_window_width_slots": float(window_width.mean()),
        "assignment_density_proxy": float((window_width * jobs_df["gpu_count_required"]).sum()),
    }


def solved_cheap_block_utilization(cluster_results: pd.DataFrame, config: DifficultyConfig) -> float:
    """Return aggregate GPU utilization inside the cheap block from solved results."""

    block = cluster_results[
        (cluster_results["hour"] >= config.cheap_block_start)
        & (cluster_results["hour"] < config.cheap_block_end)
    ]
    if block.empty or "cluster_gpu_load" not in block.columns:
        return 0.0
    return float(block["cluster_gpu_load"].sum() / block["gpu_capacity"].sum())


def summarize_cluster_utilization(cluster_results: pd.DataFrame) -> dict[str, float]:
    """Summarize resource utilization across all clusters and slots."""

    metrics: dict[str, float] = {}
    specs = {
        "power": ("cluster_load", "capacity"),
        "gpu": ("cluster_gpu_load", "gpu_capacity"),
        "cpu": ("cluster_cpu_load", "cpu_capacity"),
        "memory": ("cluster_memory_load", "memory_capacity_gb"),
    }
    for label, (load_col, capacity_col) in specs.items():
        if load_col not in cluster_results.columns or capacity_col not in cluster_results.columns:
            continue
        utilization = (cluster_results[load_col] / cluster_results[capacity_col].replace(0, np.nan)).fillna(0.0)
        metrics[f"max_{label}_utilization"] = float(utilization.max())
        metrics[f"mean_{label}_utilization"] = float(utilization.mean())
    return metrics


## Solve and Classify

The classification we care most about is `optimization_hard`: Gurobi found a feasible solution, but the optimality gap remains meaningful or proving optimality took nontrivial time.


In [ ]:
STATUS_NAMES = {
    GRB.OPTIMAL: "optimal",
    GRB.TIME_LIMIT: "time_limit",
    GRB.SUBOPTIMAL: "suboptimal",
    GRB.INFEASIBLE: "infeasible",
    GRB.INF_OR_UNBD: "infeasible_or_unbounded",
    GRB.UNBOUNDED: "unbounded",
}


def optimize_with_first_solution_tracking(model) -> tuple[bool, float | None]:
    """Optimize a Gurobi model and track time to first incumbent solution."""

    model.Params.TimeLimit = TIME_LIMIT_SECONDS
    model.Params.MIPGap = MIP_GAP
    model._first_solution_s = None

    def callback(cb_model, where):
        if where == GRB.Callback.MIPSOL and cb_model._first_solution_s is None:
            cb_model._first_solution_s = float(cb_model.cbGet(GRB.Callback.RUNTIME))

    model.optimize(callback)
    has_solution = model.SolCount > 0
    return has_solution, model._first_solution_s


def classify_run(status: str, gurobi_status: str | None, runtime_s: float | None, mip_gap: float | None, first_solution_s: float | None) -> str:
    """Classify whether a generated instance is easy, infeasible, or optimization-hard."""

    if status != "solved_or_feasible":
        if gurobi_status in {"infeasible", "infeasible_or_unbounded"}:
            return "infeasible"
        return "failed_or_no_solution"
    if mip_gap is not None and mip_gap > MIP_GAP:
        return "optimization_hard_gap"
    if gurobi_status == "time_limit":
        return "optimization_hard_time_limit"
    if runtime_s is not None and runtime_s > 10.0:
        return "optimization_hard_runtime"
    if first_solution_s is not None and runtime_s is not None and first_solution_s < runtime_s * 0.25 and runtime_s > 3.0:
        return "proof_harder_than_feasibility"
    return "easy_or_medium"


def solve_difficulty_instance(config: DifficultyConfig, seed: int, scenario_id: int) -> dict[str, Any]:
    """Generate, solve, and summarize one combinatorial difficulty instance."""

    row: dict[str, Any] = {"scenario_id": scenario_id, "seed": seed, **asdict(config)}
    try:
        jobs_df, hidden_df = generate_difficulty_instance(config, seed, clusters_df)
        diagnostics = instance_diagnostics(jobs_df, config, clusters_df)
        hourly_df, model_config = build_energy_inputs(config, clusters_df)
        row.update(diagnostics)

        build_start = time.time()
        model, variables = build_milp_model(
            jobs_df,
            hourly_df,
            clusters_df,
            model_config,
            model_name=f"difficulty_{scenario_id}_{seed}",
            enforce_gpu_constraints=True,
            enforce_cpu_constraints=True,
            enforce_memory_constraints=True,
        )
        build_s = time.time() - build_start

        has_solution, first_solution_s = optimize_with_first_solution_tracking(model)
        gurobi_status = STATUS_NAMES.get(model.Status, str(model.Status))
        if not has_solution:
            row.update(
                {
                    "status": "failed",
                    "gurobi_status": gurobi_status,
                    "classification": classify_run("failed", gurobi_status, None, None, first_solution_s),
                    "build_s": build_s,
                    "first_solution_s": first_solution_s,
                    "num_vars": int(model.NumVars),
                    "num_constraints": int(model.NumConstrs),
                    "assignment_vars": len(variables["x"]),
                }
            )
            return row

        hourly_results = extract_hourly_results(hourly_df, variables)
        cluster_results = extract_cluster_hourly_results(variables)
        schedule_df = extract_schedule(jobs_df, variables)
        summary = compute_summary_metrics(hourly_results, model_config)
        utilization = summarize_cluster_utilization(cluster_results)
        runtime_s = float(model.Runtime)
        mip_gap = float(model.MIPGap)

        row.update(
            {
                "status": "solved_or_feasible",
                "gurobi_status": gurobi_status,
                "classification": classify_run("solved_or_feasible", gurobi_status, runtime_s, mip_gap, first_solution_s),
                "build_s": build_s,
                "runtime_s": runtime_s,
                "first_solution_s": first_solution_s,
                "mip_gap": mip_gap,
                "objective": float(model.ObjVal),
                "best_bound": float(model.ObjBound),
                "num_vars": int(model.NumVars),
                "num_constraints": int(model.NumConstrs),
                "assignment_vars": len(variables["x"]),
                "cheap_block_gpu_utilization": solved_cheap_block_utilization(cluster_results, config),
                **summary,
                **utilization,
            }
        )

        if SAVE_SELECTED_SOLUTIONS and row["classification"] != "easy_or_medium":
            run_dir = OUTPUT_DIR / f"scenario_{scenario_id:03d}_seed_{seed}"
            run_dir.mkdir(parents=True, exist_ok=True)
            jobs_df.to_csv(run_dir / "jobs.csv", index=False)
            hidden_df.to_csv(run_dir / "hidden_feasible_schedule.csv", index=False)
            hourly_df.to_csv(run_dir / "hourly.csv", index=False)
            clusters_df.to_csv(run_dir / "clusters.csv", index=False)
            schedule_df.to_csv(run_dir / "schedule.csv", index=False)
            hourly_results.to_csv(run_dir / "hourly_results.csv", index=False)
            cluster_results.to_csv(run_dir / "cluster_hourly_results.csv", index=False)
    except Exception as exc:
        row.update(
            {
                "status": "failed",
                "error_type": type(exc).__name__,
                "error": str(exc),
                "classification": "generation_or_build_failed",
            }
        )
    return row


## Run Sweep

This sweep is intentionally small enough to run repeatedly. Increase `REPLICATIONS`, add more knob values, or tighten `TIME_LIMIT_SECONDS` after you identify a promising difficult region.


In [ ]:
rows = []
for scenario_id, config in enumerate(SWEEP_CONFIGS):
    for rep in range(REPLICATIONS):
        seed = BASE_SEED + scenario_id * 1_000 + rep
        print(
            f"scenario={scenario_id:03d} rep={rep} "
            f"util={config.target_gpu_utilization} pressure={config.cheap_block_pressure} "
            f"restricted={config.restricted_gpu_share} multi={config.multi_gpu_share}"
        )
        result = solve_difficulty_instance(config, seed, scenario_id)
        rows.append(result)
        print(
            {
                key: result.get(key)
                for key in [
                    "status",
                    "classification",
                    "num_jobs",
                    "runtime_s",
                    "first_solution_s",
                    "mip_gap",
                    "assignment_vars",
                    "global_gpu_slot_utilization",
                    "cheap_block_pressure_actual",
                    "cheap_block_gpu_utilization",
                    "max_gpu_utilization",
                ]
            }
        )

results_df = pd.DataFrame(rows)
results_path = OUTPUT_DIR / "combinatorial_difficulty_results.csv"
results_df.to_csv(results_path, index=False)
results_path, results_df.head()


## Aggregate and Inspect Difficulty Regions


In [ ]:
if results_df.empty:
    raise ValueError("Run the sweep first.")

summary_columns = [
    "target_gpu_utilization",
    "cheap_block_pressure",
    "restricted_gpu_share",
    "multi_gpu_share",
]

agg = (
    results_df.assign(solved=lambda df: df["status"].eq("solved_or_feasible"))
    .groupby(summary_columns, dropna=False)
    .agg(
        runs=("seed", "count"),
        solved_runs=("solved", "sum"),
        median_num_jobs=("num_jobs", "median"),
        median_assignment_vars=("assignment_vars", "median"),
        median_runtime_s=("runtime_s", "median"),
        p90_runtime_s=("runtime_s", lambda s: s.dropna().quantile(0.90) if s.notna().any() else np.nan),
        median_mip_gap=("mip_gap", "median"),
        median_first_solution_s=("first_solution_s", "median"),
        median_global_gpu_utilization=("global_gpu_slot_utilization", "median"),
        median_cheap_pressure=("cheap_block_pressure_actual", "median"),
        median_cheap_block_gpu_utilization=("cheap_block_gpu_utilization", "median"),
    )
    .reset_index()
)
agg["solve_rate"] = agg["solved_runs"] / agg["runs"]
agg.to_csv(OUTPUT_DIR / "combinatorial_difficulty_aggregate.csv", index=False)
agg.sort_values(["median_runtime_s", "median_mip_gap"], ascending=False).head(20)


In [ ]:
classification_counts = (
    results_df.groupby(["target_gpu_utilization", "cheap_block_pressure", "restricted_gpu_share", "multi_gpu_share", "classification"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["target_gpu_utilization", "cheap_block_pressure", "count"], ascending=[True, True, False])
)
classification_counts.head(30)


## Plots


In [ ]:
import matplotlib.pyplot as plt

plot_df = results_df[results_df["status"] == "solved_or_feasible"].copy()
if plot_df.empty:
    raise ValueError("No solved runs available to plot.")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

scatter = axes[0].scatter(
    plot_df["cheap_block_pressure_actual"],
    plot_df["runtime_s"],
    c=plot_df["multi_gpu_share_actual"],
    s=60,
)
axes[0].set_title("Runtime vs Cheap-Block Pressure")
axes[0].set_xlabel("Actual cheap-block pressure")
axes[0].set_ylabel("Runtime (s)")
fig.colorbar(scatter, ax=axes[0], label="Multi-GPU share")

axes[1].scatter(plot_df["assignment_vars"], plot_df["runtime_s"], s=60)
axes[1].set_title("Runtime vs Binary Assignment Variables")
axes[1].set_xlabel("x[j,k,s] variables")
axes[1].set_ylabel("Runtime (s)")

axes[2].scatter(plot_df["cheap_block_gpu_utilization"], plot_df["mip_gap"], s=60)
axes[2].set_title("MIP Gap vs Cheap-Block Utilization")
axes[2].set_xlabel("Solved cheap-block GPU utilization")
axes[2].set_ylabel("MIP gap")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

pivot = agg.pivot_table(
    index="target_gpu_utilization",
    columns="cheap_block_pressure",
    values="median_runtime_s",
    aggfunc="median",
)
fig, ax = plt.subplots(figsize=(7, 4))
image = ax.imshow(pivot.values, aspect="auto", origin="lower")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{value:.1f}" for value in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f"{value:.2f}" for value in pivot.index])
ax.set_xlabel("Cheap-block pressure target")
ax.set_ylabel("Global GPU utilization target")
ax.set_title("Median Runtime Heatmap")
fig.colorbar(image, ax=ax, label="seconds")
plt.tight_layout()
plt.show()


## Candidate Hard Instances

These are the first cases to inspect for quantum-friendly benchmarks: solved instances with nontrivial runtime, nonzero gap, or a large difference between time to first feasible solution and total solve time.


In [ ]:
candidate_hard = results_df[
    results_df["classification"].isin(
        [
            "optimization_hard_gap",
            "optimization_hard_time_limit",
            "optimization_hard_runtime",
            "proof_harder_than_feasibility",
        ]
    )
].copy()

candidate_hard.sort_values(["runtime_s", "mip_gap"], ascending=False).head(20)


## Interpretation

A useful quantum-oriented instance should be small enough to encode later, but not trivial for Gurobi. Prefer cases where:

- `assignment_vars` is small or moderate,
- Gurobi finds a feasible solution,
- `mip_gap` remains nonzero or runtime is meaningfully above the easy baseline,
- cheap-block pressure is above 1 but global GPU utilization is below 1,
- multi-GPU jobs are present, creating packing conflicts.


## Focused Strict-Optimality Rerun

The first sweep found a promising region where Gurobi usually finds feasible solutions quickly but sometimes needs much longer to close the bound. This focused rerun searches that region with a stricter optimality tolerance and a longer time limit.

Changes from the broad sweep:

- `MIP_GAP = 0.001` instead of `0.01`.
- Longer time limit.
- Higher `max_jobs` so high-utilization settings are less likely to stop early.
- Knob values concentrated around the harder region from the first run.

This section writes separate CSVs and does not overwrite the broad-sweep results.


In [ ]:
FOCUSED_OUTPUT_DIR = OUTPUT_DIR / "focused_strict_gap"
FOCUSED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FOCUSED_TIME_LIMIT_SECONDS = 180
FOCUSED_MIP_GAP = 0.001
FOCUSED_REPLICATIONS = 3
FOCUSED_BASE_SEED = 50_001

FOCUSED_CONFIGS = [
    DifficultyConfig(
        target_gpu_utilization=util,
        cheap_block_pressure=pressure,
        restricted_gpu_share=restricted,
        multi_gpu_share=multi,
        max_jobs=90,
        window_min=4,
        window_max=10,
    )
    for util, pressure, restricted, multi in product(
        [0.80, 0.85, 0.90],
        [1.40, 1.70, 2.00],
        [0.25, 0.50],
        [0.30, 0.50],
    )
]

len(FOCUSED_CONFIGS)


In [ ]:
def run_focused_strict_sweep() -> pd.DataFrame:
    """Run the focused strict-gap sweep without modifying broad-sweep outputs."""

    global TIME_LIMIT_SECONDS, MIP_GAP, SAVE_SELECTED_SOLUTIONS

    original_time_limit = TIME_LIMIT_SECONDS
    original_mip_gap = MIP_GAP
    original_save_selected = SAVE_SELECTED_SOLUTIONS

    TIME_LIMIT_SECONDS = FOCUSED_TIME_LIMIT_SECONDS
    MIP_GAP = FOCUSED_MIP_GAP
    SAVE_SELECTED_SOLUTIONS = False

    focused_rows = []
    try:
        for focused_id, config in enumerate(FOCUSED_CONFIGS):
            for rep in range(FOCUSED_REPLICATIONS):
                seed = FOCUSED_BASE_SEED + focused_id * 1_000 + rep
                print(
                    f"focused={focused_id:03d} rep={rep} "
                    f"util={config.target_gpu_utilization} pressure={config.cheap_block_pressure} "
                    f"restricted={config.restricted_gpu_share} multi={config.multi_gpu_share}"
                )
                result = solve_difficulty_instance(config, seed, focused_id)
                result["focused_id"] = focused_id
                result["time_limit_seconds"] = FOCUSED_TIME_LIMIT_SECONDS
                result["target_mip_gap"] = FOCUSED_MIP_GAP
                focused_rows.append(result)
                print(
                    {
                        key: result.get(key)
                        for key in [
                            "status",
                            "gurobi_status",
                            "classification",
                            "num_jobs",
                            "runtime_s",
                            "first_solution_s",
                            "mip_gap",
                            "assignment_vars",
                            "global_gpu_slot_utilization",
                            "cheap_block_pressure_actual",
                            "global_utilization_target_met",
                            "cheap_pressure_target_met",
                        ]
                    }
                )
    finally:
        TIME_LIMIT_SECONDS = original_time_limit
        MIP_GAP = original_mip_gap
        SAVE_SELECTED_SOLUTIONS = original_save_selected

    focused_df = pd.DataFrame(focused_rows)
    focused_df.to_csv(FOCUSED_OUTPUT_DIR / "focused_strict_gap_results.csv", index=False)
    return focused_df


focused_results_df = run_focused_strict_sweep()
focused_results_df.head()


In [ ]:
if focused_results_df.empty:
    raise ValueError("Run the focused strict-gap sweep first.")

focused_summary_columns = [
    "target_gpu_utilization",
    "cheap_block_pressure",
    "restricted_gpu_share",
    "multi_gpu_share",
]

focused_agg = (
    focused_results_df.assign(solved=lambda df: df["status"].eq("solved_or_feasible"))
    .groupby(focused_summary_columns, dropna=False)
    .agg(
        runs=("seed", "count"),
        solved_runs=("solved", "sum"),
        median_num_jobs=("num_jobs", "median"),
        median_assignment_vars=("assignment_vars", "median"),
        median_runtime_s=("runtime_s", "median"),
        p90_runtime_s=("runtime_s", lambda s: s.dropna().quantile(0.90) if s.notna().any() else np.nan),
        max_runtime_s=("runtime_s", "max"),
        median_mip_gap=("mip_gap", "median"),
        max_mip_gap=("mip_gap", "max"),
        median_first_solution_s=("first_solution_s", "median"),
        median_global_gpu_utilization=("global_gpu_slot_utilization", "median"),
        median_cheap_pressure=("cheap_block_pressure_actual", "median"),
        target_hit_rate=("global_utilization_target_met", "mean"),
        cheap_pressure_hit_rate=("cheap_pressure_target_met", "mean"),
        median_cheap_block_gpu_utilization=("cheap_block_gpu_utilization", "median"),
    )
    .reset_index()
)
focused_agg["solve_rate"] = focused_agg["solved_runs"] / focused_agg["runs"]
focused_agg.to_csv(FOCUSED_OUTPUT_DIR / "focused_strict_gap_aggregate.csv", index=False)
focused_agg.sort_values(["median_runtime_s", "max_mip_gap"], ascending=False).head(20)


In [ ]:
focused_classification_counts = (
    focused_results_df.groupby(
        [
            "target_gpu_utilization",
            "cheap_block_pressure",
            "restricted_gpu_share",
            "multi_gpu_share",
            "classification",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="count")
    .sort_values(["target_gpu_utilization", "cheap_block_pressure", "count"], ascending=[True, True, False])
)
focused_classification_counts.head(40)


In [ ]:
focused_candidate_hard = focused_results_df[
    focused_results_df["classification"].isin(
        [
            "optimization_hard_gap",
            "optimization_hard_time_limit",
            "optimization_hard_runtime",
            "proof_harder_than_feasibility",
        ]
    )
].copy()

focused_candidate_hard.sort_values(["runtime_s", "mip_gap"], ascending=False).head(25)


## Focused Rerun Plots


In [ ]:
import matplotlib.pyplot as plt

focused_plot_df = focused_results_df[focused_results_df["status"] == "solved_or_feasible"].copy()
if focused_plot_df.empty:
    raise ValueError("No solved focused runs available to plot.")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

scatter = axes[0].scatter(
    focused_plot_df["global_gpu_slot_utilization"],
    focused_plot_df["runtime_s"],
    c=focused_plot_df["cheap_block_pressure_actual"],
    s=70,
)
axes[0].set_title("Runtime vs Actual Global Utilization")
axes[0].set_xlabel("Actual global GPU-slot utilization")
axes[0].set_ylabel("Runtime (s)")
fig.colorbar(scatter, ax=axes[0], label="Actual cheap pressure")

axes[1].scatter(focused_plot_df["assignment_vars"], focused_plot_df["runtime_s"], s=70)
axes[1].set_title("Runtime vs Assignment Variables")
axes[1].set_xlabel("x[j,k,s] variables")
axes[1].set_ylabel("Runtime (s)")

axes[2].scatter(focused_plot_df["first_solution_s"], focused_plot_df["runtime_s"], s=70)
axes[2].set_title("Proof Time vs Feasibility Time")
axes[2].set_xlabel("Time to first feasible solution (s)")
axes[2].set_ylabel("Total runtime (s)")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

heatmap = focused_agg.pivot_table(
    index="target_gpu_utilization",
    columns="cheap_block_pressure",
    values="median_runtime_s",
    aggfunc="median",
)
fig, ax = plt.subplots(figsize=(7, 4))
image = ax.imshow(heatmap.values, aspect="auto", origin="lower")
ax.set_xticks(range(len(heatmap.columns)))
ax.set_xticklabels([f"{value:.1f}" for value in heatmap.columns])
ax.set_yticks(range(len(heatmap.index)))
ax.set_yticklabels([f"{value:.2f}" for value in heatmap.index])
ax.set_xlabel("Cheap-block pressure target")
ax.set_ylabel("Global GPU utilization target")
ax.set_title("Focused Strict-Gap Median Runtime")
fig.colorbar(image, ax=ax, label="seconds")
plt.tight_layout()
plt.show()


## Create Constrained Benchmark Instances

This section materializes reusable benchmark instances from the constrained region identified in the focused strict-gap experiment. These are not another sweep; they are a curated instance pack intended for follow-up MILP, QUBO, QAOA, PCE, and hybrid tests.

The selected region is:

- high global GPU pressure: `target_gpu_utilization` around `0.90`,
- high restricted compatibility: `restricted_gpu_share = 0.50`,
- meaningful multi-GPU packing conflicts: `multi_gpu_share = 0.30` or `0.50`,
- moderate requested cheap-block pressure, because actual pressure already overshoots the target.

Each exported instance contains:

- `jobs.csv`,
- `hourly.csv`,
- `clusters.csv`,
- `hidden_feasible_schedule.csv`,
- `metadata.json`.


In [ ]:
BENCHMARK_OUTPUT_DIR = OUTPUT_DIR / "constrained_benchmark_instances"
BENCHMARK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONSTRAINED_BENCHMARK_CONFIGS = [
    {
        "label": "high_util_restricted_multi03_pressure14",
        "config": DifficultyConfig(
            target_gpu_utilization=0.90,
            cheap_block_pressure=1.40,
            restricted_gpu_share=0.50,
            multi_gpu_share=0.30,
            max_jobs=90,
            window_min=4,
            window_max=10,
        ),
    },
    {
        "label": "high_util_restricted_multi05_pressure14",
        "config": DifficultyConfig(
            target_gpu_utilization=0.90,
            cheap_block_pressure=1.40,
            restricted_gpu_share=0.50,
            multi_gpu_share=0.50,
            max_jobs=90,
            window_min=4,
            window_max=10,
        ),
    },
    {
        "label": "high_util_restricted_multi03_pressure17",
        "config": DifficultyConfig(
            target_gpu_utilization=0.90,
            cheap_block_pressure=1.70,
            restricted_gpu_share=0.50,
            multi_gpu_share=0.30,
            max_jobs=90,
            window_min=4,
            window_max=10,
        ),
    },
    {
        "label": "high_util_restricted_multi05_pressure17",
        "config": DifficultyConfig(
            target_gpu_utilization=0.90,
            cheap_block_pressure=1.70,
            restricted_gpu_share=0.50,
            multi_gpu_share=0.50,
            max_jobs=90,
            window_min=4,
            window_max=10,
        ),
    },
    {
        "label": "high_util_restricted_multi03_pressure20",
        "config": DifficultyConfig(
            target_gpu_utilization=0.90,
            cheap_block_pressure=2.00,
            restricted_gpu_share=0.50,
            multi_gpu_share=0.30,
            max_jobs=90,
            window_min=4,
            window_max=10,
        ),
    },
    {
        "label": "slightly_lower_util_restricted_multi05_pressure20",
        "config": DifficultyConfig(
            target_gpu_utilization=0.85,
            cheap_block_pressure=2.00,
            restricted_gpu_share=0.50,
            multi_gpu_share=0.50,
            max_jobs=90,
            window_min=4,
            window_max=10,
        ),
    },
]

BENCHMARK_SEEDS = [91_001, 91_002, 91_003]
len(CONSTRAINED_BENCHMARK_CONFIGS), len(BENCHMARK_SEEDS)


In [ ]:
def safe_json_value(value: Any) -> Any:
    """Convert numpy/pandas scalar values into JSON-serializable Python values."""

    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if value is pd.NA:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return value


def write_benchmark_instance(
    *,
    label: str,
    config: DifficultyConfig,
    seed: int,
    instance_index: int,
) -> dict[str, Any]:
    """Generate and persist one reusable constrained benchmark instance."""

    instance_name = f"{instance_index:03d}_{label}_seed_{seed}"
    instance_dir = BENCHMARK_OUTPUT_DIR / instance_name
    instance_dir.mkdir(parents=True, exist_ok=True)

    jobs_df, hidden_df = generate_difficulty_instance(config, seed, clusters_df)
    hourly_df, model_config = build_energy_inputs(config, clusters_df)
    diagnostics = instance_diagnostics(jobs_df, config, clusters_df)

    jobs_df.to_csv(instance_dir / "jobs.csv", index=False)
    hourly_df.to_csv(instance_dir / "hourly.csv", index=False)
    clusters_df.to_csv(instance_dir / "clusters.csv", index=False)
    hidden_df.to_csv(instance_dir / "hidden_feasible_schedule.csv", index=False)

    metadata = {
        "instance_name": instance_name,
        "label": label,
        "seed": seed,
        "slot_minutes": SLOT_MINUTES,
        "horizon_slots": HORIZON_SLOTS,
        "delta_t": DELTA_T,
        "difficulty_config": asdict(config),
        "model_config": asdict(model_config),
        "diagnostics": diagnostics,
        "files": {
            "jobs": "jobs.csv",
            "hourly": "hourly.csv",
            "clusters": "clusters.csv",
            "hidden_feasible_schedule": "hidden_feasible_schedule.csv",
        },
    }
    metadata = json.loads(json.dumps(metadata, default=safe_json_value))
    (instance_dir / "metadata.json").write_text(json.dumps(metadata, indent=2))

    return {
        "instance_name": instance_name,
        "instance_dir": str(instance_dir.relative_to(PROJECT_ROOT)),
        "label": label,
        "seed": seed,
        **diagnostics,
    }


def create_constrained_benchmark_instances() -> pd.DataFrame:
    """Create the curated constrained benchmark instance pack."""

    manifest_rows = []
    instance_index = 0
    for item in CONSTRAINED_BENCHMARK_CONFIGS:
        label = item["label"]
        config = item["config"]
        for seed in BENCHMARK_SEEDS:
            row = write_benchmark_instance(
                label=label,
                config=config,
                seed=seed,
                instance_index=instance_index,
            )
            manifest_rows.append(row)
            instance_index += 1

    manifest_df = pd.DataFrame(manifest_rows)
    manifest_df.to_csv(BENCHMARK_OUTPUT_DIR / "manifest.csv", index=False)
    return manifest_df


benchmark_manifest_df = create_constrained_benchmark_instances()
benchmark_manifest_df


## Optional: Solve Benchmark Instances

Run this optional cell when you want to verify and rank the exported benchmark pack with the stricter Gurobi settings. It is separate from instance creation so that the instance pack can be generated quickly and reused by quantum experiments without rerunning Gurobi every time.


In [ ]:
def solve_exported_benchmark_instance(instance_dir: Path, time_limit: float = 180, mip_gap: float = 0.001) -> dict[str, Any]:
    """Solve one exported benchmark instance and return compact Gurobi/resource metrics."""

    jobs_df = pd.read_csv(instance_dir / "jobs.csv")
    hourly_df = pd.read_csv(instance_dir / "hourly.csv")
    clusters = pd.read_csv(instance_dir / "clusters.csv")
    metadata = json.loads((instance_dir / "metadata.json").read_text())
    model_config = ModelConfig(**metadata["model_config"])

    model, variables = build_milp_model(
        jobs_df,
        hourly_df,
        clusters,
        model_config,
        model_name=f"benchmark_{instance_dir.name}",
        enforce_gpu_constraints=True,
        enforce_cpu_constraints=True,
        enforce_memory_constraints=True,
    )

    original_time_limit = TIME_LIMIT_SECONDS
    original_mip_gap = MIP_GAP
    try:
        globals()["TIME_LIMIT_SECONDS"] = time_limit
        globals()["MIP_GAP"] = mip_gap
        has_solution, first_solution_s = optimize_with_first_solution_tracking(model)
    finally:
        globals()["TIME_LIMIT_SECONDS"] = original_time_limit
        globals()["MIP_GAP"] = original_mip_gap

    result: dict[str, Any] = {
        "instance_name": instance_dir.name,
        "status": "failed" if not has_solution else "solved_or_feasible",
        "gurobi_status": STATUS_NAMES.get(model.Status, str(model.Status)),
        "first_solution_s": first_solution_s,
        "runtime_s": float(model.Runtime),
        "num_vars": int(model.NumVars),
        "num_constraints": int(model.NumConstrs),
        "assignment_vars": len(variables["x"]),
    }
    if has_solution:
        hourly_results = extract_hourly_results(hourly_df, variables)
        cluster_results = extract_cluster_hourly_results(variables)
        summary = compute_summary_metrics(hourly_results, model_config)
        utilization = summarize_cluster_utilization(cluster_results)
        result.update(
            {
                "mip_gap": float(model.MIPGap),
                "objective": float(model.ObjVal),
                "best_bound": float(model.ObjBound),
                "cheap_block_gpu_utilization": solved_cheap_block_utilization(
                    cluster_results,
                    DifficultyConfig(**metadata["difficulty_config"]),
                ),
                **summary,
                **utilization,
            }
        )
    return result


# Set RUN_BENCHMARK_SOLVE = True when you want to solve and rank the exported pack.
RUN_BENCHMARK_SOLVE = False

if RUN_BENCHMARK_SOLVE:
    benchmark_solve_rows = []
    for instance_dir in sorted(BENCHMARK_OUTPUT_DIR.glob("[0-9][0-9][0-9]_*/")):
        print(f"Solving {instance_dir.name}")
        benchmark_solve_rows.append(solve_exported_benchmark_instance(instance_dir))
    benchmark_solve_df = pd.DataFrame(benchmark_solve_rows)
    benchmark_solve_df.to_csv(BENCHMARK_OUTPUT_DIR / "benchmark_solve_results.csv", index=False)
else:
    benchmark_solve_df = pd.DataFrame()

benchmark_solve_df.sort_values(["runtime_s", "mip_gap"], ascending=False).head(20) if not benchmark_solve_df.empty else "Set RUN_BENCHMARK_SOLVE = True to solve exported instances."
